In [1]:
import pandas as pd
import numpy as np 
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error

In [5]:
Q_com = pd.read_excel("Q_Commerce_Data_rating_Count.xlsx")
Q_com
Cluster = pd.read_excel("Clustering_5.xlsx")
Cluster

,Brand,Quantity ( Grams),MRP,Price,Price/100 Grams,Discount (%),Cluster
0,Farmley,70.0,79,79,112.857143,0,7
1,Farmley,33.0,30,29,87.878788,3,7
2,Farmley,20.0,40,27,135.000000,32,9
3,Farmley,200.0,400,339,169.500000,15,1
4,Farmley,70.0,79,79,112.857143,0,7
...,...,...,...,...,...,...,...
443,Healthy Master,250.0,394,301,120.400000,24,3
444,Healthy Master,100.0,141,109,109.000000,23,6
445,Healthy Master,100.0,141,109,109.000000,23,6
446,Healthy Master,30.0,33,30,100.000000,9,7


In [ ]:
Q_com.columns = Q_com.columns.str.strip()
Cluster.columns = Cluster.columns.str.strip()


In [8]:
Q_com.columns

Index(['Brand', 'Quantity', 'MRP', 'Price', 'Price_100_g', 'Discount',
       'Rating', 'Rating_Count'],
      dtype='object')

In [9]:
Cluster.columns

Index(['Brand', 'Quantity ( Grams)', 'MRP', 'Price', 'Price/100 Grams',
       'Discount (%)', 'Cluster'],
      dtype='object')

In [11]:
Cluster.rename(columns={'Quantity ( Grams)':'Quantity',
           'Discount (%)': 'Discount'            

}, inplace=True)

In [12]:
Cluster

,Brand,Quantity,MRP,Price,Price/100 Grams,Discount,Cluster
0,Farmley,70.0,79,79,112.857143,0,7
1,Farmley,33.0,30,29,87.878788,3,7
2,Farmley,20.0,40,27,135.000000,32,9
3,Farmley,200.0,400,339,169.500000,15,1
4,Farmley,70.0,79,79,112.857143,0,7
...,...,...,...,...,...,...,...
443,Healthy Master,250.0,394,301,120.400000,24,3
444,Healthy Master,100.0,141,109,109.000000,23,6
445,Healthy Master,100.0,141,109,109.000000,23,6
446,Healthy Master,30.0,33,30,100.000000,9,7


In [13]:
features = [ 'Quantity', 'MRP','Discount']
target = 'Rating_Count'

In [16]:
Q_com_features = Q_com[features].copy()
Q_com_target = Q_com[target].copy()
Q_com_features
Q_com_target

0      846
1     1300
2     4000
3     1900
4     1300
      ... 
70     121
71     192
72     242
73     202
74     195
Name: Rating_Count, Length: 75, dtype: int64

In [18]:
X = Q_com_features
y = Q_com_target

In [19]:
X.fillna(X.median(),inplace=True)
y.fillna(y.median(),inplace=True)

In [21]:
X_train , X_test , y_train , y_test = train_test_split(X,y , test_size=0.2 , random_state=42)
model = xgb.XGBRegressor(
    objective= "reg:squarederror",
    n_estimator = 100,
    learning_rate = 0.1,
    max_depth = 5,
    random_state=42

)

model.fit(X_train,y_train)
y_pred= model.predict(X_test)


C:\Users\Ritwik Singh\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [19:49:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [44]:
rmse = (root_mean_squared_error(y_test,y_pred))
mae = mean_absolute_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)*100


In [45]:
print(f"RSME score (Root Mean Square Error) = {rmse:.2f}")
print(f"MAE score (Mean Absolute Error)= {mae:.2f}")
print(f"r2 score = {r2:.2f} %")

RSME score (Root Mean Square Error) = 2317.59
MAE score (Mean Absolute Error)= 1694.85
r2 score = 85.67 %


In [37]:
cluster_features = Cluster[features]
cluster_features
cluster_features.fillna(cluster_features.median(),inplace=True)

C:\Users\Ritwik Singh\AppData\Local\Temp\ipykernel_13592\3222767797.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cluster_features.fillna(cluster_features.median(),inplace=True)


In [40]:
predicted_ratings = model.predict(cluster_features)
predicted_ratings = np.maximum(0,predicted_ratings)
predicted_ratings = np.round(predicted_ratings).astype(int)

In [41]:
Cluster["projected_rating_count"] = predicted_ratings

In [42]:
Cluster

,Brand,Quantity,MRP,Price,Price/100 Grams,Discount,Cluster,projected_rating_count
0,Farmley,70.0,79,79,112.857143,0,7,1404
1,Farmley,33.0,30,29,87.878788,3,7,1575
2,Farmley,20.0,40,27,135.000000,32,9,14107
3,Farmley,200.0,400,339,169.500000,15,1,270
4,Farmley,70.0,79,79,112.857143,0,7,1404
...,...,...,...,...,...,...,...,...
443,Healthy Master,250.0,394,301,120.400000,24,3,265
444,Healthy Master,100.0,141,109,109.000000,23,6,319
445,Healthy Master,100.0,141,109,109.000000,23,6,319
446,Healthy Master,30.0,33,30,100.000000,9,7,259


In [43]:
Cluster.to_excel("projected_rating_count_cummulative.xlsx",index=False)